# 02 - Fraud Analysis
Analyse approfondie des patterns de fraude sur le dataset Kartik.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

ROOT = Path('c:/Users/hp/Desktop/BigData2.0/fraud-detection-platform')
train = pd.read_csv(ROOT / 'data/raw/fraudTrain.csv')
test  = pd.read_csv(ROOT / 'data/raw/fraudTest.csv')
df    = pd.concat([train, test], ignore_index=True)

df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour']        = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['month']       = df['trans_date_trans_time'].dt.month
df['distance']    = np.sqrt((df['lat'] - df['merch_lat'])**2 + (df['long'] - df['merch_long'])**2) * 111
df['dob']         = pd.to_datetime(df['dob'])
df['age']         = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

fraud  = df[df['is_fraud'] == 1]
normal = df[df['is_fraud'] == 0]
print(f'Fraudes : {len(fraud):,} | Normal : {len(normal):,}')

## 1. Comparaison des montants

In [ ]:
print('=== MONTANTS - FRAUDE ===')
print(fraud['amt'].describe().round(2))
print('\n=== MONTANTS - NORMAL ===')
print(normal['amt'].describe().round(2))

stat, p = stats.mannwhitneyu(fraud['amt'], normal['amt'], alternative='two-sided')
print(f'\nTest Mann-Whitney p-value = {p:.2e} → {"Différence significative" if p < 0.05 else "Pas de différence"}')

## 2. Distance domicile-commerçant

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column='distance', by='is_fraud', ax=axes[0],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='crimson', linewidth=2))
axes[0].set_title('Distance domicile-commerçant')
axes[0].set_xlabel('is_fraud')
axes[0].set_ylabel('Distance (km)')
plt.sca(axes[0]); plt.title('Distance domicile-commerçant')

normal['distance'].clip(upper=300).hist(ax=axes[1], bins=50, alpha=0.6, label='Normal', color='steelblue', density=True)
fraud['distance'].clip(upper=300).hist(ax=axes[1], bins=50, alpha=0.6, label='Fraude', color='crimson', density=True)
axes[1].set_title('Distribution de la distance')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Densité')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Distance médiane Fraude : {fraud["distance"].median():.1f} km')
print(f'Distance médiane Normal : {normal["distance"].median():.1f} km')

## 3. Analyse par catégorie

In [ ]:
cat_stats = (df.groupby('category')
               .agg(total=('is_fraud','count'),
                    fraudes=('is_fraud','sum'),
                    amt_moyen=('amt','mean'))
               .assign(fraud_rate=lambda x: (x['fraudes']/x['total']*100).round(3))
               .sort_values('fraud_rate', ascending=False))
display(cat_stats.round(2))

## 4. Analyse démographique (genre, âge)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraude par genre
gender_fraud = df.groupby('gender')['is_fraud'].mean() * 100
gender_fraud.plot(kind='bar', ax=axes[0], color=['steelblue','crimson'], alpha=0.8, rot=0)
axes[0].set_title('Taux de fraude par genre')
axes[0].set_ylabel('Taux de fraude (%)')

# Fraude par tranche d'âge
df['age_group'] = pd.cut(df['age'], bins=[0,25,35,45,55,65,100],
                          labels=['<25','25-35','35-45','45-55','55-65','65+'])
age_fraud = df.groupby('age_group')['is_fraud'].mean() * 100
age_fraud.plot(kind='bar', ax=axes[1], color='darkorange', alpha=0.8, rot=0)
axes[1].set_title('Taux de fraude par tranche d\'âge')
axes[1].set_ylabel('Taux de fraude (%)')

plt.tight_layout()
plt.show()

## 5. Feature Engineering exploratoire

In [ ]:
df['amt_log']            = np.log1p(df['amt'])
df['distance_log']       = np.log1p(df['distance'])
df['is_night']           = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
df['city_pop_log']       = np.log1p(df['city_pop'])

high_risk = ['shopping_net','misc_net','grocery_pos','shopping_pos']
df['high_risk_category'] = df['category'].isin(high_risk).astype(int)

features = ['amt_log','distance_log','hour','is_night','city_pop_log','high_risk_category','age']
corr_fraud = (df[features + ['is_fraud']]
              .corr()['is_fraud']
              .drop('is_fraud')
              .sort_values(key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['crimson' if v > 0 else 'steelblue' for v in corr_fraud.values]
corr_fraud.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Corrélation des features avec is_fraud')
ax.set_xlabel('Corrélation de Pearson')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()
print(corr_fraud)

## 6. Agrégations par carte (cc_num)

In [ ]:
df_sorted = df.sort_values(['cc_num','trans_date_trans_time'])

# Nombre de transactions cumulées par carte
df_sorted['tx_count_cumul'] = df_sorted.groupby('cc_num').cumcount() + 1

agg = df_sorted.groupby('is_fraud')[['tx_count_cumul','amt']].mean().round(2)
print('=== MOYENNES PAR CLASSE ===')
display(agg)

# Top 10 cartes les plus fraudées
top_cards = (fraud.groupby('cc_num')['is_fraud']
             .count().sort_values(ascending=False).head(10))
print('\n=== TOP 10 CARTES FRAUDÉES ===')
print(top_cards)

## 7. Synthèse

| Feature | Type | Corrélation is_fraud |
|---|---|---|
| `amt_log` | Numérique | forte |
| `distance_log` | Numérique | forte |
| `hour` | Numérique | modérée |
| `is_night` | Binaire | modérée |
| `category` | Catégoriel | forte |
| `city_pop_log` | Numérique | modérée |
| `age` | Numérique | faible |

→ Spécifications complètes dans `src/features/feature_list.txt`